# Validation du model turbulent poreux en TrioCFD - Cas 1

In [ ]:
from trustutils import run 
# import all the packages
from trustutils.jupyter import run
from trustutils import plot, files as tf
from datetime import date

from string import Template 
import re
import numpy as np
import sympy as sp
import pandas as pd
import os
import math
import csv

import matplotlib.pyplot as plt
from IPython.display import display, Markdown as md
run.introduction("ALi Fadlallah")

run.TRUST_parameters()

### Description: 

Rub reference case and the test cases

In [ ]:
channel_width  = 1.0 # meter
channel_height = 3.0 # meter

def compute_geometry(porosity, channel_width=channel_width, channel_height=channel_height):
    
    Lx1 = round(channel_width * (1 - porosity)/3.0, 6)
    Lx3 = Lx1
    Lx2 = round(channel_width - (Lx1 + Lx3), 6)

    
    Ly_PE = round(channel_height/12, 6)
    Ly_bas = round(3*Ly_PE, 6)
    Ly_haut = round(channel_height - (Ly_PE+Ly_bas), 6)
    
    # X positions
    X1 = round(Lx1, 6)
    X2 = round(X1 + Lx2, 6)
    X3 = round(X2 + Lx3, 6)

    # Y positions
    Y1 = round(2*Ly_PE, 6)
    Y2 = round(Y1 + Ly_PE, 6)
    Y3 = round(Y2 + Ly_PE, 6)
    Y4 = round(Y3 + Ly_PE, 6)
    #Y5 = round(Y2 + Ly, 6)
    
    return {
        "X1": X1, "X2": X2, "X3": X3,
        "Y1": Y1, "Y2": Y2, "Y3": Y3, "Y4": Y4,
        "Lx1": Lx1, "Lx2": Lx2, "Lx3": Lx3,
        "Ly_PE": Ly_PE, "Ly_bas": Ly_bas, "Ly_haut":Ly_haut
    }



# Node counts
Nx1 = 30
Nx2 = 35 #35
Nx3 = 30
Ny =  160 #160
Ny_PE = 50

velocities = np.array([2.7])
porosities = np.array([0.3, 0.5, 0.9])



run.reset()
run.defaultSuite_ = run.TRUSTSuite(runPrepare=False)

for v_in in velocities:
    for phi in porosities:
        geometry = compute_geometry(phi)
        geometry.update({ "v_in": v_in,
                          "Nx1": Nx1, "Nx2": Nx2, "Nx3": Nx3,
                          "Ny": Ny, "Ny_PE": Ny_PE, "channel_width":channel_width, "channel_height":channel_height,})
        
        run.addCaseFromTemplate("jdd_ref.data", f"reference/vin_{v_in:.2g}/phi_{phi:.1g}", geometry)

run.printCases()

In [ ]:
import itertools


Nx = 80
Ny = 60

#parameters for Frottement_poreux


# Define parameter ranges

c0_values = np.array([5.0e-7, 6.0e-7, 7.0e-7, 8.0e-7, 9.0e-7, 1.0e-6, 5.0e-6])
c1_values = np.array([6.0e4])


# Loop over all combinations
for phi, c0, c1, v_in in itertools.product(porosities, c0_values, c1_values, velocities):
    #case_name = f"c0_{c0:.1g}_c1_{c1:.1g}/phi_{phi:.1f}"
    
    geometry = compute_geometry(phi)
    run.addCaseFromTemplate(
        "jdd.data", f"validation/vin_{v_in:.2g}/c0_{c0:.1g}_c1_{c1:.1g}/phi_{phi:.1g}",
        {   # Frottement poreux
            "longueur": geometry["Lx2"],
            
            "c_0": c0,
            "c_1": c1,
            "v_in": v_in,

            "Nx": Nx,
            "Ny": Ny,
            
            "Nx_PE_bas": Nx,
            "Ny_PE_bas": Ny,
            
            "Nx_PE": Nx,
            "Ny_PE": 2*Ny,

            "Nx_PE_haut": Nx,
            "Ny_PE_haut": Ny,
            
           
            "porosity": phi,
            "channel_width": channel_width,
            "channel_height": channel_height,

            # Geometry coordinates
            "X1": geometry["X1"],
            "X2": geometry["X2"],
            "X3": geometry["X3"],
            
            #"Y_PE_bas": geometry["Y1"]/2.0,
            "Y1": geometry["Y1"],
            #"Y_PE_haut": geometry["Y1"] + geometry["Y1"]/2.0 ,
            "Y2": geometry["Y2"],
            "Y3": geometry["Y3"],
            "Y4": geometry["Y4"],

            # Sizes
            "Lx": channel_width,
            "Lx1": geometry["Lx1"],
            "Lx3": geometry["Lx3"],
            
            "Ly_PE_bas": round(geometry["Ly_PE"]/3.0 , 6),
            "Ly_PE_haut": round(geometry["Ly_PE"]/3.0 , 6),
            "Ly_PE": geometry["Ly_PE"],
            "Ly_bas": geometry["Ly_bas"] - geometry["Ly_PE"] ,
            "Ly_haut": geometry["Ly_haut"] - geometry["Ly_PE"] ,

            "solid":round(1-phi,6),
        }
    )

run.printCases()
run.runCases()
run.tablePerf()

#visit.showMesh(f"validation/vin_{v_in:.1g}c0_{c0:.1g}_c1_{c1:.1g}/phi_0.3/VenaContracta.lata","dom")
#visit.showMesh(f"reference/vin_{v_in:.1g}/phi_0.3/VenaContracta.lata","dom")

In [ ]:
def GetSonData(path):
    if not os.path.exists(path):
        raise ValueError(f"The path ({path}) does not exist!")
    # On plot le dernier instant
    donne = tf.SonSEGFile(path, None)
    
    # On recupere le nom des variables, leur dimension et la coordonnee que l'on veut tracer
    ncompo = donne.getnCompo()
    entries = donne.getEntries()
    t = donne.getValues(entries[0])[0]

    ## find closest value of t
    idx = -1
    X = donne.getXAxis()
    Y = np.zeros([len(entries[0::ncompo]), ncompo])
    
    for k in range(ncompo):
        for n, i in enumerate(entries[k::ncompo]):
            Y[n, k] = list(donne.getValues(i)[1])[idx]
    X = np.array(X)
    #print(Y)
    Y = np.array(Y)
    

    return X, Y

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Error computation function using L-norms (normalized by reference)
def compute_L_norm_errors(ref, val):
    epsilon = 1e-10  # To avoid division by zero

    # L1 norm (Mean Relative Error)
    L1_rel = np.mean(np.abs((ref - val) / (ref + epsilon)))

    # L2 norm (Root Mean Square Error, normalized)
    L2_rel = np.sqrt(np.mean(((ref - val) / (ref + epsilon)) ** 2))

    # L-infinity norm (Maximum Relative Error)
    Linf_rel = np.max(np.abs((ref - val) / (ref + epsilon)))

    return L1_rel, L2_rel, Linf_rel

In [ ]:
import itertools
from collections import defaultdict

# Structure: {"SONDE_VBAS" : {(c0, c1): {"L1_rel": [...], "L2_rel": [...], "Linf_rel": [...]}}}
errors_by_case = defaultdict(
    lambda: defaultdict(
        lambda: {"L1_rel": [], "L2_rel": [], "Linf_rel": []}
    )
)


# Sonde names and labels
sondes_labeled = {"SONDE_PRESSION2": "Pressure", "SONDE_VHAUT": "V_haut",
                  "SONDE_VBAS": "V_bas", "SONDE_VCHANNEL": "V_channel",}


for c0, c1, v_in in itertools.product(c0_values, c1_values, velocities):
    for phi in porosities:
        for sonde, label in sondes_labeled.items():
            # File paths
            path_ref = os.path.join(f"{run.BUILD_DIRECTORY}/reference/vin_{v_in:.2g}/phi_{phi:.1g}/", f"jdd_ref_{sonde}.son")
            path_val = os.path.join(f"{run.BUILD_DIRECTORY}/validation/vin_{v_in:.2g}/c0_{c0:.1g}_c1_{c1:.1g}/phi_{phi:.1g}/", f"jdd_{sonde}.son")
            
            if sonde != 'SONDE_PRESSION2' : 
                try:
                    _, V_ref = GetSonData(path_ref)
                    # compute the norm i.e. sqrt(Vx^2 + Vy^2)
                    V_ref = np.linalg.norm(V_ref.reshape(-1, 2), axis=1)
        
                    _, V_val = GetSonData(path_val)
                    V_val = np.linalg.norm(V_val.reshape(-1, 2), axis=1)
                
                except Exception as e:
                    print(f"Error reading or processing: phi={phi}, c0={c0}, c1={c1}, sonde={sonde}\n{e}")
                    continue
                
                # Error computation
                L1_rel, L2_rel, Linf_rel = compute_L_norm_errors(V_ref, V_val)
            else:
                try:
                    _, P_ref = GetSonData(path_ref)
                    _, P_val = GetSonData(path_val)

                    delta_P_ref = np.abs(P_ref - P_ref[0])/P_ref[0]
                    delta_P_val = np.abs(P_val - P_val[0])/P_val[0]
                except Exception as e:
                    print(f"Error reading or processing: phi={phi}, c0={c0}, c1={c1}, sonde={sonde}\n{e}")
                    continue
                
                # Error computation
                L1_rel, L2_rel, Linf_rel = compute_L_norm_errors(delta_P_ref, delta_P_val)
            
            # Store with tuple key
            key = (c0, c1)
            errors_by_case[sonde][key]["L1_rel"].append(L1_rel)
            errors_by_case[sonde][key]["L2_rel"].append(L2_rel)
            errors_by_case[sonde][key]["Linf_rel"].append(Linf_rel)

In [ ]:
import pandas as pd
from cycler import cycler
from matplotlib.lines import Line2D

def find_error(sonde):
    records = []
    for (c0, c1), error_dict in errors_by_case[sonde].items():
        records.append({
            "c0": c0,
            "c1": c1,
            "L1_mean": np.mean(error_dict["L1_rel"]),
            "L1_max": np.max(error_dict["L1_rel"]),
            "L2_mean": np.mean(error_dict["L2_rel"]),
            "L2_max": np.max(error_dict["L2_rel"]),
            "Linf_mean": np.mean(error_dict["Linf_rel"]),
            "Linf_max": np.max(error_dict["Linf_rel"]),
        })
    
    error_df = pd.DataFrame(records)

    pivot_tables = {}
    for metric in ["L1_mean", "L2_mean", "Linf_max"]:
        pivot_tables[metric] = error_df.pivot(index="c1", columns="c0", values=metric)

    # Create pivot tables for each metric (e.g., L1_mean)
    pivot_L1_mean = error_df.pivot(index="c1", columns="c0", values="L1_mean")
    pivot_L2_mean = error_df.pivot(index="c1", columns="c0", values="L2_mean")
    pivot_Linf_max = error_df.pivot(index="c1", columns="c0", values="Linf_max")

    return error_df, pivot_tables

def plot_heatmap(sonde):

    _, pivot_tables, = find_error(sonde)
    
    #for metric in ["L1_mean", "L2_mean", "Linf_max"]:
    for metric in ["L1_mean", "L2_mean"]:
        pivot = pivot_tables[metric]
        plt.figure(figsize=(8, 6))
        im = plt.imshow(pivot.values, cmap="viridis", origin='lower', aspect='auto')
        
        # Tick labels
        plt.xticks(ticks=range(len(pivot.columns)), labels=[f"{v:.1g}" for v in pivot.columns])
        plt.yticks(ticks=range(len(pivot.index)), labels=[f"{v:.1g}" for v in pivot.index])
        
        plt.xlabel("c0")
        plt.ylabel("c1")
        plt.title(f"{metric} error for {sondes_labeled[sonde]}")
        cbar = plt.colorbar(im)
        cbar.set_label("Normalized error")
    
        plt.tight_layout()
        #plt.savefig(f"heatmap_case2_c0_{sondes_labeled[sonde]}_{metric}.jpg", dpi=400)
        plt.show()

def plot_best_worst_error(sonde):

    error_df,_ = find_error(sonde)

    # Extract best and worst for 3 error norms
    best_L1 = error_df.loc[error_df["L1_mean"].idxmin()][["c0", "c1"]]
    worst_L1 = error_df.loc[error_df["L1_mean"].idxmax()][["c0", "c1"]]
    
    best_L2 = error_df.loc[error_df["L2_mean"].idxmin()][["c0", "c1"]]
    worst_L2 = error_df.loc[error_df["L2_mean"].idxmax()][["c0", "c1"]]
    
    best_Linf = error_df.loc[error_df["Linf_max"].idxmin()][["c0", "c1"]]
    worst_Linf = error_df.loc[error_df["Linf_max"].idxmax()][["c0", "c1"]]
    
    # Bundle all
    keys_to_plot = {
        "L1_mean Best": (best_L1["c0"], best_L1["c1"]),
        "L1_mean Worst": (worst_L1["c0"], worst_L1["c1"]),
        "L2_mean Best": (best_L2["c0"], best_L2["c1"]),
        "L2_mean Worst": (worst_L2["c0"], worst_L2["c1"]),
        "Linf_max Best": (best_Linf["c0"], best_Linf["c1"]),
        "Linf_max Worst": (worst_Linf["c0"], worst_Linf["c1"]),
    }

    # plot parameters
    solid_frac = [1 - phi for phi in porosities]

    # Color per norm
    color_map = {
        "L1_rel": "tab:blue",
        "L2_rel": "tab:green",
        "Linf_rel": "tab:red"
    }

    plt.figure(figsize=(10, 6))

    for label, key in keys_to_plot.items():
        if key not in errors_by_case[sonde]:
            continue

        # Determine error type
        if "L1" in label:
            norm = "L1_rel"
        elif "L2" in label:
            norm = "L2_rel"
        elif "Linf" in label:
            norm = "Linf_rel"
        else:
            norm = "L2_rel"  # Default for fallback

        # Visual styling
        if "Best" in label:
            linestyle = "-"
            marker = "x"
        elif "Worst" in label:
            linestyle = "--"
            marker = ""
        else:  # Default config
            linestyle = ":"
            marker = "s"

        # c0, c1 values
        c0_str = f"{key[0]:.1g}"
        c1_str = f"{key[1]:.1g}"
        label_full = f"{label} (c0={c0_str}, c1={c1_str})"

        y = errors_by_case[sonde][key][norm]

        plt.plot(
            solid_frac,
            y,
            linestyle=linestyle,
            marker=marker,
            label=label_full,
            color=color_map[norm]
        )

    # Labels and format
    plt.xlabel(r"$1-\phi$", fontsize=12)
    plt.ylabel("Normalized Relative Error", fontsize=12)
    plt.title(f"Error Norm Profiles for {sondes_labeled[sonde]}", fontsize=14)
    plt.yscale("log")
    #plt.gca().yaxis.set_major_formatter(ticker.ScalarFormatter(useMathText=True))
    #plt.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))

    plt.grid(True, which="both", linestyle="--", alpha=0.4)
    plt.legend(fontsize=9)
    plt.tight_layout()
    #plt.savefig("best_error.jpg", dpi=400)
    plt.show()

def plot_sonde(sonde):

    error_df,_ = find_error(sonde)
    # Extract best and worst for 3 error norms
    best_L1 = error_df.loc[error_df["L1_mean"].idxmin()][["c0", "c1"]]
    worst_L1 = error_df.loc[error_df["L1_mean"].idxmax()][["c0", "c1"]]
    
    best_L2 = error_df.loc[error_df["L2_mean"].idxmin()][["c0", "c1"]]
    worst_L2 = error_df.loc[error_df["L2_mean"].idxmax()][["c0", "c1"]]
    
    best_Linf = error_df.loc[error_df["Linf_max"].idxmin()][["c0", "c1"]]
    worst_Linf = error_df.loc[error_df["Linf_max"].idxmax()][["c0", "c1"]]


    colors = plt.cm.tab10(np.linspace(0, 1, len(porosities)))
    plt.figure(figsize=(10,8), dpi=400)
    plt.grid()
    
    #for c0, c1 in itertools.product(c0_values, c1_values):
    c0, c1 = best_L2["c0"], best_L2["c1"]    

    for v_in in velocities:
        plt.title(rf"{sondes_labeled[sonde]} for $v_{{in}}={v_in}$")
        for i, phi in enumerate(porosities):
            # File paths
            path_ref = os.path.join(f"{run.BUILD_DIRECTORY}/reference/vin_{v_in:.2g}/phi_{phi:.1g}/", f"jdd_ref_{sonde}.son")
            path_val = os.path.join(f"{run.BUILD_DIRECTORY}/validation/vin_{v_in:.2g}/c0_{c0:.1g}_c1_{c1:.1g}/phi_{phi:.1g}/", f"jdd_{sonde}.son")
                
            
            if sonde == 'SONDE_PRESSION2' : 
                Y_ref, P_ref = GetSonData(path_ref)
                Y_val, P_val = GetSonData(path_val)
    
                P_ref = P_ref[:, 0]
                P_val = P_val[:, 0]
                
                delta_P_ref = np.abs((P_ref[0]- P_ref)/P_ref[0])
                delta_P_val = np.abs((P_val[0]- P_val)/P_val[0])
    
                plt.plot(Y_ref, delta_P_ref ,marker='*', color=colors[i], label=rf"$\phi_{{ref}}={phi:.1g}$")
                plt.plot(Y_val, delta_P_val ,":", color=colors[i], label=rf"$\phi_{{vald}}={phi:.1g}$")

                plt.title(rf"Pressure loss for $v_{{in}}={v_in}$")
                plt.ylabel(rf'$\Delta P$ [Pa] ')
                plt.xlabel('Y [m]')
                
            else: 
                X, data_ref = GetSonData(path_ref)
                # compute the norm i.e. sqrt(Vx^2 + Vy^2)
                V_ref = np.linalg.norm(data_ref.reshape(-1, 2), axis=1)
    
                X, data_val = GetSonData(path_val)
                V_val = np.linalg.norm(data_val.reshape(-1, 2), axis=1)
    
                plt.plot(X, V_ref ,marker='*', color=colors[i], label=rf"$\phi_{{ref}}={phi:.1g}$")
                plt.plot(X, V_val ,":", color=colors[i], label=rf"$\phi_{{vald}}={phi:.1g}$")
                
                # x and y label
                plt.title(rf"{sondes_labeled[sonde]} for $v_{{in}}={v_in}$")
                plt.xlabel('X [m]')
                plt.ylabel('V [m/s]')
            
            plt.ticklabel_format(useOffset=False,style='plain')
            plt.legend()
        plt.tight_layout()
        #plt.savefig("sonde_pression.pdf", dpi=400)
        plt.show()

In [ ]:
sondes_labeled = {"SONDE_PRESSION2": "Pressure", "SONDE_VHAUT": "V_haut",
                  "SONDE_VBAS": "V_bas", "SONDE_VCHANNEL": "V_channel",}

for sonde, label in sondes_labeled.items():
    plot_heatmap(sonde)
    plot_best_worst_error(sonde)
    plot_sonde(sonde)

# Error Evaluation

To evaluate the difference between the **validation model** and the **reference solution**, we compute dimensionless error metrics based on normalized L-norms.

## Normalized $L_1$ Norm (Mean Relative Error)

This metric quantifies the **average** relative error across all data points.

$$
L_1 = \frac{1}{n} \sum_{i=1}^{n} \left| \frac{\text{val}_i - \text{ref}_i}{\text{ref}_i} \right|
$$

## Normalized $L_2$ Norm (Root Mean Square Relative Error)

This norm emphasizes **larger deviations**, i.e. outliers.

$$
L_2 = \sqrt{ \frac{1}{n} \sum_{i=1}^{n} \left( \frac{\text{val}_i - \text{ref}_i}{\text{ref}_i} \right)^2 }
$$

## Normalized $L_\infty$ Norm (Maximum Relative Error)

This gives the **worst-case relative error**, i.e. largest deviation.

$$
L_\infty = \max_{i} \left| \frac{\text{val}_i - \text{ref}_i}{\text{ref}_i} \right|
$$
